# Stokes/Correlation Polarization-Basis Conversion for Images

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/imaging/demo_stokes_correlation_conversion.ipynb)

This notebook demonstrates how to convert astronomical **image** data between the
feed/correlation basis (correlation products) and the **Stokes** basis using the
`transform_polarization_basis` processing function.

**Key Concepts:**
- **Correlation products** represent the raw output from interferometer feeds
  (e.g. `XX, XY, YX, YY` for linear feeds or `RR, RL, LR, LL` for circular feeds).
- **Stokes parameters** (`I, Q, U, V`) represent physical polarization properties.

**This tutorial covers:**
1. Linear polarization conversion (`XX, XY, YX, YY` ↔ `I, Q, U, V`)
2. Circular polarization conversion (`RR, RL, LR, LL` ↔ `I, Q, U, V`)
3. Round-trip conversion verification
4. Working with multidimensional image cubes
5. Inspecting the raw transformation matrix with `get_transformation_matrix`
6. Coordinate/attribute preservation and a worked image example

> **Note on scope.** Earlier versions of this tutorial used the removed
> `corr_to_stokes` / `stokes_to_corr` helpers, which operated on bare NumPy
> *visibility* arrays. The current API, `transform_polarization_basis`, operates on
> **xarray image datasets** (an `xarray.Dataset` with a `polarization` dimension) and
> is the same function used inside the AstroVIPER imaging pipeline. This tutorial has
> therefore been adapted to the image domain; the underlying linear algebra
> (the polarization mixing matrices) is identical to the visibility-domain case.

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    os.system("pip install --upgrade astroviper[all]")

    import astroviper  # noqa: F401 -- availability probe for the pip-install fallback

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

## Imports

In [ ]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt

from astroviper.processing_functions.image_analysis.transform_polarization_basis import (
    get_transformation_matrix,
    transform_polarization_basis,
)

# Set random seed for reproducibility
np.random.seed(42)

### A small helper

`transform_polarization_basis` expects an `xarray.Dataset` that has a `polarization`
dimension (the AstroVIPER image layout is
`(time, frequency, polarization, l, m)`). The helper below wraps a NumPy array in a
minimal image dataset so the examples stay compact.

In [ ]:
def make_image_dataset(pol_data, pol_labels, dims=None):
    """Wrap a polarization-bearing array in a minimal AstroVIPER-style image Dataset.

    Parameters
    ----------
    pol_data : np.ndarray
        Array whose axes match ``dims``
        (default ``(time, frequency, polarization, l, m)``).
    pol_labels : list of str
        Polarization coordinate labels, e.g. ['XX', 'XY', 'YX', 'YY'] or
        ['I', 'Q', 'U', 'V'].
    """
    if dims is None:
        dims = ["time", "frequency", "polarization", "l", "m"]
    return xr.Dataset(
        {"SKY": (dims, np.asarray(pol_data))},
        coords={"polarization": list(pol_labels)},
    )

## 1. Linear Polarization: Correlation Products ↔ Stokes Parameters

Linear feeds produce correlation products in the order **[XX, XY, YX, YY]**.
`transform_polarization_basis` (basis `"stokes"`) converts them to Stokes
**[I, Q, U, V]** with:

- I = (XX + YY) / 2
- Q = (XX − YY) / 2
- U = (XY + YX) / 2
- V = i (YX − XY) / 2

and converts back (basis `"linear"`) with the inverse:

- XX = I + Q
- XY = U + iV
- YX = U − iV
- YY = I − Q

### Build a synthetic linear-correlation image

We use a single pixel with known properties so the conversion is easy to verify.

In [ ]:
# A source with:
#   Total intensity   I = 10.0 Jy
#   Linear pol        Q =  2.0 Jy   (more power in XX than YY)
#   No 45-deg linear pol (U = 0) and no circular pol (V = 0)
#
# Using the inverse formulas above:
#   XX = I + Q = 12.0,  XY = U + iV = 0,  YX = U - iV = 0,  YY = I - Q = 8.0
linear_corr = np.array([12.0 + 0j, 0j, 0j, 8.0 + 0j]).reshape(1, 1, 4, 1, 1)

img_linear = make_image_dataset(linear_corr, ["XX", "XY", "YX", "YY"])

print("Input polarization:", list(img_linear.polarization.values))
print("SKY correlation values [XX, XY, YX, YY]:")
print(img_linear["SKY"].values.ravel())

### Convert correlation products to Stokes parameters

In [ ]:
# transform_polarization_basis modifies its input in place when overwrite=True
# (the default), so we pass a deep copy to keep the original for comparison.
img_stokes = transform_polarization_basis(
    img_linear.copy(deep=True), new_polarization_basis="stokes"
)

stokes_linear = img_stokes["SKY"].values.ravel()
print("Output polarization:", list(img_stokes.polarization.values))
print("Stokes parameters [I, Q, U, V]:", stokes_linear)
print("Expected: [10.+0j, 2.+0j, 0.+0j, 0.+0j]")
print("Match:", np.allclose(stokes_linear, [10.0, 2.0, 0.0, 0.0]))

### Convert Stokes parameters back to correlation products

In [ ]:
# Stokes -> linear correlations (basis "linear")
img_linear_rt = transform_polarization_basis(
    img_stokes.copy(deep=True), new_polarization_basis="linear"
)

print("Round-trip polarization:", list(img_linear_rt.polarization.values))
print("Round-trip [XX, XY, YX, YY]:", img_linear_rt["SKY"].values.ravel())
print("Original   [XX, XY, YX, YY]:", img_linear["SKY"].values.ravel())
print(
    "Round-trip successful:",
    np.allclose(img_linear["SKY"].values, img_linear_rt["SKY"].values),
)

## 2. Circular Polarization: Correlation Products ↔ Stokes Parameters

Circular feeds produce correlation products in the order **[RR, RL, LR, LL]**.
`transform_polarization_basis` (basis `"stokes"`) converts them to Stokes
**[I, Q, U, V]** with:

- I = (RR + LL) / 2
- Q = (RL + LR) / 2
- U = i (LR − RL) / 2
- V = (RR − LL) / 2

and converts back (basis `"circular"`) with the inverse:

- RR = I + V
- RL = Q + iU
- LR = Q − iU
- LL = I − V

### Build a synthetic circular-correlation image

In [ ]:
# A source with:
#   Total intensity   I = 8.0 Jy
#   No linear pol (Q = 0, U = 0)
#   Circular pol      V = 2.0 Jy   (more right- than left-hand)
#
# Using the inverse formulas above:
#   RR = I + V = 10.0,  RL = Q + iU = 0,  LR = Q - iU = 0,  LL = I - V = 6.0
circular_corr = np.array([10.0 + 0j, 0j, 0j, 6.0 + 0j]).reshape(1, 1, 4, 1, 1)

img_circular = make_image_dataset(circular_corr, ["RR", "RL", "LR", "LL"])

print("Input polarization:", list(img_circular.polarization.values))
print("SKY correlation values [RR, RL, LR, LL]:")
print(img_circular["SKY"].values.ravel())

### Convert correlation products to Stokes parameters

In [ ]:
img_stokes_c = transform_polarization_basis(
    img_circular.copy(deep=True), new_polarization_basis="stokes"
)

stokes_circular = img_stokes_c["SKY"].values.ravel()
print("Output polarization:", list(img_stokes_c.polarization.values))
print("Stokes parameters [I, Q, U, V]:", stokes_circular)
print("Expected: [8.+0j, 0.+0j, 0.+0j, 2.+0j]")
print("Match:", np.allclose(stokes_circular, [8.0, 0.0, 0.0, 2.0]))

### Convert Stokes parameters back to correlation products

In [ ]:
img_circular_rt = transform_polarization_basis(
    img_stokes_c.copy(deep=True), new_polarization_basis="circular"
)

print("Round-trip polarization:", list(img_circular_rt.polarization.values))
print("Round-trip [RR, RL, LR, LL]:", img_circular_rt["SKY"].values.ravel())
print("Original   [RR, RL, LR, LL]:", img_circular["SKY"].values.ravel())
print(
    "Round-trip successful:",
    np.allclose(img_circular["SKY"].values, img_circular_rt["SKY"].values),
)

## 3. Multidimensional Image Cubes

Real image data are multidimensional: `(time, frequency, polarization, l, m)`.
`transform_polarization_basis` locates the `polarization` dimension **by name**, so
the other axes can be of any size and in any position — only the polarization axis is
mixed. (uv-grid, point-spread-function, and airy-disk primary-beam variables are
automatically skipped, so a full imaging dataset can be transformed in a single call.)

In [ ]:
# A realistic image cube: (time=1, frequency=8, polarization=4, l=64, m=64)
n_time, n_freq, n_pol, n_l, n_m = 1, 8, 4, 64, 64

cube = np.random.randn(n_time, n_freq, n_pol, n_l, n_m) + 1j * np.random.randn(
    n_time, n_freq, n_pol, n_l, n_m
)

img_cube = make_image_dataset(cube, ["XX", "XY", "YX", "YY"])
print("Input image cube sizes:", dict(img_cube.sizes))
print("Polarization:", list(img_cube.polarization.values))

In [ ]:
# Convert the whole cube to Stokes
img_cube_stokes = transform_polarization_basis(
    img_cube.copy(deep=True), new_polarization_basis="stokes"
)
print("Stokes cube sizes:", dict(img_cube_stokes.sizes))
print("Polarization:", list(img_cube_stokes.polarization.values))
print("Shape is preserved; only the polarization axis is mixed.")

In [ ]:
# Round-trip: Stokes -> linear correlations
img_cube_rt = transform_polarization_basis(
    img_cube_stokes.copy(deep=True), new_polarization_basis="linear"
)
max_err = np.max(np.abs(img_cube["SKY"].values - img_cube_rt["SKY"].values))
print("Round-trip polarization:", list(img_cube_rt.polarization.values))
print(
    "Round-trip successful:",
    np.allclose(img_cube["SKY"].values, img_cube_rt["SKY"].values),
)
print("Maximum absolute error:", max_err)

## 4. Inspecting the Raw Transformation Matrix

`get_transformation_matrix` returns the underlying mixing matrix together with the
ordered input and output polarization labels. `transform_polarization_basis` uses
exactly this matrix internally
(`result[out] = Σ_i matrix[out, i] · data[i]`), so you can reproduce the conversion
by hand.

In [ ]:
matrix, in_labels, out_labels = get_transformation_matrix(
    ["XX", "XY", "YX", "YY"], new_polarization_basis="stokes"
)
print("Input  (columns):", in_labels)
print("Output (rows)   :", out_labels)
print("Transformation matrix (rows = output, cols = input):")
print(np.array2string(matrix, precision=2, suppress_small=True))

In [ ]:
# Apply the matrix by hand to the Section-1 correlation vector and compare
corr_vec = np.array([12.0 + 0j, 0j, 0j, 8.0 + 0j])  # [XX, XY, YX, YY]
stokes_by_hand = matrix @ corr_vec
print("Stokes by matrix multiply:", stokes_by_hand)
print(
    "Matches transform_polarization_basis:", np.allclose(stokes_by_hand, stokes_linear)
)

In [ ]:
# The circular correlation -> Stokes matrix
matrix_c, in_c, out_c = get_transformation_matrix(
    ["RR", "RL", "LR", "LL"], new_polarization_basis="stokes"
)
print("Input  (columns):", in_c)
print("Output (rows)   :", out_c)
print(np.array2string(matrix_c, precision=2, suppress_small=True))

## 5. Coordinate/Attribute Preservation and a Worked Image

`transform_polarization_basis` operates on an `xarray.Dataset`: it preserves every
non-polarization coordinate and all dataset attributes, updating only the
`polarization` coordinate labels. Below we build a small Stokes sky model
(a Gaussian source with linear polarization), convert it to the linear correlation
basis (as a linear-feed instrument would record it), convert it back to Stokes,
verify the round-trip, and plot the Stokes maps.

In [ ]:
# Gaussian source, 64 x 64, single time/channel
npix = 64
yy, xx = np.mgrid[0:npix, 0:npix]
gauss = np.exp(-(((xx - 32) ** 2 + (yy - 32) ** 2) / (2 * 8.0**2)))

I = 10.0 * gauss
Q = 0.20 * I  # 20% linear polarization along the Q axis
U = 0.05 * I
V = np.zeros_like(I)

# Stokes model with layout (time=1, frequency=1, polarization=4, l, m)
stokes_model = np.stack([I, Q, U, V])[None, None].astype(complex)

sky_stokes = xr.Dataset(
    {"SKY": (["time", "frequency", "polarization", "l", "m"], stokes_model)},
    coords={
        "time": [0.0],
        "frequency": [1.4e9],
        "polarization": ["I", "Q", "U", "V"],
        "l": np.arange(npix),
        "m": np.arange(npix),
    },
    attrs={"telescope": "VLA", "field": "MySource", "units": "Jy/pixel"},
)
print(sky_stokes)

In [ ]:
# Stokes -> linear correlations (what a linear-feed instrument records)
sky_linear = transform_polarization_basis(
    sky_stokes.copy(deep=True), new_polarization_basis="linear"
)
print("Correlation polarization:", list(sky_linear.polarization.values))

# linear correlations -> Stokes (recover the sky)
sky_stokes_rt = transform_polarization_basis(
    sky_linear.copy(deep=True), new_polarization_basis="stokes"
)
print("Recovered polarization:  ", list(sky_stokes_rt.polarization.values))
print("Attributes preserved:    ", sky_stokes_rt.attrs)
print(
    "Non-pol coords preserved:",
    "frequency" in sky_stokes_rt.coords and "l" in sky_stokes_rt.coords,
)
print(
    "Round-trip successful:   ",
    np.allclose(sky_stokes["SKY"].values, sky_stokes_rt["SKY"].values),
)

In [ ]:
# Plot the recovered Stokes maps
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, pol in zip(axes, ["I", "Q", "U", "V"], strict=False):
    plane = sky_stokes_rt["SKY"].sel(polarization=pol).values[0, 0].real
    im = ax.imshow(plane, origin="lower", cmap="inferno")
    ax.set_title(f"Stokes {pol}")
    ax.set_xlabel("l")
    ax.set_ylabel("m")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Recovered Stokes maps after a correlation-basis round-trip")
fig.tight_layout()
plt.show()